In [1]:
import torch
from PIL import Image
from transformers import ViTForImageClassification, ViTImageProcessor
import time
import os

In [5]:
# 1. 저장된 모델 경로 설정 (📌 학습 후 저장된 폴더 이름과 일치해야 합니다)
MODEL_PATH = "./"

# 2. 추론할 이미지 파일 경로 설정 (📌 실제 이미지 파일로 변경하세요)
IMAGE_FILE_PATH = "pets.jpg"

In [6]:
# -----------------------------------------------------
# 모델 및 프로세서 로드
# -----------------------------------------------------
try:
    if not os.path.isdir(MODEL_PATH):
        raise FileNotFoundError(f"모델 폴더를 찾을 수 없습니다: {MODEL_PATH}")

    processor = ViTImageProcessor.from_pretrained(MODEL_PATH)
    model = ViTForImageClassification.from_pretrained(MODEL_PATH)

    # GPU 사용 가능 여부 확인 후 디바이스 설정 (CPU 환경을 가정)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval() # 추론 모드로 설정

    print(f"✅ 모델 로드 완료. 현재 Device: {device}")

except FileNotFoundError as e:
    print(f"❌ 오류: {e}")
    print("모델 저장 폴더와 파일(pytorch_model.bin, config.json 등)의 위치를 확인해주세요.")
    exit()
except Exception as e:
    print(f"❌ 모델 로드 중 예기치 않은 오류 발생: {e}")
    exit()

✅ 모델 로드 완료. 현재 Device: cpu


In [7]:
# -----------------------------------------------------
# 추론 실행
# -----------------------------------------------------
try:
    # 이미지 로드 (모델 전처리 요구사항에 맞춰 RGB로 변환)
    start_time = time.time()
    image = Image.open(IMAGE_FILE_PATH).convert("RGB")

    # 4. 이미지 전처리 및 텐서 생성
    # processor는 저장된 설정(Resize, Normalize 등)에 따라 이미지를 변환합니다.
    inputs = processor(images=image, return_tensors="pt").to(device)

    # 5. 예측 (Prediction) 수행
    with torch.no_grad():
        outputs = model(**inputs)

    # 6. 결과 해석
    # 로짓(logits)에서 소프트맥스를 적용하여 확률을 얻고, 가장 높은 확률의 인덱스를 찾습니다.
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_prob = probabilities.max().item()
    predicted_class_idx = probabilities.argmax(-1).item()

    # 7. 클래스 이름으로 변환
    predicted_label = model.config.id2label[predicted_class_idx]

    end_time = time.time()

    print("\n--- 추론 결과 ---")
    print(f"⏳ 소요 시간: {end_time - start_time:.4f} 초")
    print(f"🔍 예측된 클래스: **{predicted_label}**")
    print(f"📈 예측 확률: {predicted_prob:.4f}")

except FileNotFoundError:
    print(f"❌ 오류: 이미지 파일 '{IMAGE_FILE_PATH}'을(를) 찾을 수 없습니다. 경로를 다시 확인해주세요.")
except Exception as e:
    print(f"❌ 추론 과정 중 오류 발생: {e}")


--- 추론 결과 ---
⏳ 소요 시간: 0.7148 초
🔍 예측된 클래스: **pets**
📈 예측 확률: 0.5976
